#Imports

In [2]:
import sys, os
sys.path.append(os.path.abspath(".."))


In [4]:
import os, numpy as np, pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, average_precision_score, precision_recall_curve, confusion_matrix
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from src.preprocessing import AmountTimeScaler, split_X_y

DATA_PROCESSED = Path("../data/processed")
train = pd.read_csv(DATA_PROCESSED / "batch1_train.csv")
test  = pd.read_csv(DATA_PROCESSED / "batch2_test.csv")
stream1 = pd.read_csv(DATA_PROCESSED / "batch3_stream.csv")
stream2 = None
if (DATA_PROCESSED / "batch4_stream.csv").exists():
    stream2 = pd.read_csv(DATA_PROCESSED / "batch4_stream.csv")


#Fit Baseline

In [5]:
X_tr, y_tr = split_X_y(train)
X_te, y_te = split_X_y(test)

pipe = Pipeline([
    ("scale", AmountTimeScaler(scale_time=True)),
    ("clf", LogisticRegression(max_iter=500, class_weight="balanced", solver="lbfgs"))
])

pipe.fit(X_tr, y_tr)
proba_test = pipe.predict_proba(X_te)[:,1]


#Default threshold = 0.5 metrics for baseline model

In [6]:
def metrics_at_threshold(y_true, proba, thr=0.5):
    y_pred = (proba >= thr).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    ap = average_precision_score(y_true, proba)  # PR-AUC
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {"threshold":thr, "precision":p, "recall":r, "f1":f1, "pr_auc":ap,
            "TP":tp, "FP":fp, "TN":tn, "FN":fn}

baseline = metrics_at_threshold(y_te, proba_test, 0.5)
pd.DataFrame([baseline])


,threshold,precision,recall,f1,pr_auc,TP,FP,TN,FN
0,0.5,0.5,0.851064,0.629921,0.417653,40,40,9913,7


#Target-recall threshold + “manual review” workload

#Finding threshold for a target recall

In [7]:
def threshold_for_target_recall(y_true, proba, target_recall=0.80):
    precisions, recalls, thresholds = precision_recall_curve(y_true, proba)
    # precision_recall_curve returns thresholds len-1 compared to recalls
    # Find first threshold where recall >= target_recall scanning from high recall
    idx = np.where(recalls >= target_recall)[0]
    if len(idx)==0:
        # can't reach target recall; fall back to min threshold
        chosen_thr = thresholds.min() if len(thresholds)>0 else 0.0
    else:
        i = idx[-1] - 1  # align to thresholds indexing
        i = max(0, min(i, len(thresholds)-1))
        chosen_thr = thresholds[i]
    return float(chosen_thr)

target_recall = 0.80
thr = threshold_for_target_recall(y_te, proba_test, target_recall)
thr


0.9826497707982164

#Report recall & workload at that threshold

In [8]:
def review_load(y_true, proba, thr):
    y_pred = (proba >= thr).astype(int)
    flagged = y_pred.sum()
    total = len(y_true)
    return flagged, flagged/total

m = metrics_at_threshold(y_te, proba_test, thr)
flagged, frac = review_load(y_te, proba_test, thr)

print(f"Chosen threshold for ~{target_recall:.0%} recall: {thr:.4f}")
print(f"TEST metrics @thr: precision={m['precision']:.3f}, recall={m['recall']:.3f}, F1={m['f1']:.3f}, PR-AUC={m['pr_auc']:.3f}")
print(f"Manual review load: {flagged} / {len(y_te)} ({frac:.2%}) flagged")
pd.DataFrame([m])


Chosen threshold for ~80% recall: 0.9826
TEST metrics @thr: precision=0.542, recall=0.830, F1=0.655, PR-AUC=0.418
Manual review load: 72 / 10000 (0.72%) flagged


,threshold,precision,recall,f1,pr_auc,TP,FP,TN,FN
0,0.98265,0.541667,0.829787,0.655462,0.417653,39,33,9920,8


#Basic drift check

In [9]:
def evaluate_split(df, pipe, thr):
    X, y = split_X_y(df)
    proba = pipe.predict_proba(X)[:,1]
    m = metrics_at_threshold(y, proba, thr=thr)
    flagged, frac = review_load(y, proba, thr)
    m["flagged"] = flagged
    m["flagged_pct"] = frac
    return pd.Series(m)

rows = []
rows.append(evaluate_split(test, pipe, thr).rename("TEST"))
rows.append(evaluate_split(stream1, pipe, thr).rename("STREAM1"))
if stream2 is not None:
    rows.append(evaluate_split(stream2, pipe, thr).rename("STREAM2"))
pd.concat(rows, axis=1)


,TEST,STREAM1,STREAM2
threshold,0.982650,0.982650,0.982650
precision,0.541667,0.555556,0.800000
recall,0.829787,0.555556,0.800000
f1,0.655462,0.555556,0.800000
pr_auc,0.417653,0.367411,0.773928
TP,39.000000,5.000000,8.000000
FP,33.000000,4.000000,2.000000
TN,9920.000000,9987.000000,9988.000000
FN,8.000000,4.000000,2.000000
flagged,72.000000,9.000000,10.000000
